# Q33 — Two-axis parent/child 3.5 projection

## tl;dr — post-result correction

Q33 attempted to operationalize the ARA path as
`2 + (1 + half-capacity child) = 3.5`, but a post-result audit found a
coordinate error.

The endpoint relations from Q32 did **not** have half the source's raw capacity in
the common parent-facing coordinate. Their median energy-capacity ratio was
`1.27349`, producing a median path of `4.27349`.

Backward tracing did recover a strong child-pole origin: median local
`x=0.04137`, with both children at `x<=0.5` in 81.50% of source events. Their
summed realised gain/source-loss ratio had median `1.03265`.

That raw result is reproducible, but raw capacity is variable flow over ARA,
not the fixed ARA rung coordinate. Q33 also averaged two endpoint recipients
where the declared route uses one boundary-nearest child. Therefore its frozen
negative verdict is **invalid as a pure ARA 3.5 test**.

Frozen implementation verdict: **CROSS-RUNG 3.5 PROJECTION NOT SUPPORTED BY THIS IMPLEMENTATION**

Independent validation: **PASS**.


## Context

Q30 tested a different `1.5` proxy. Q32 independently normalized every
relation onto its own local `0–2`, which was suitable for finding ordered
handover but erased cross-rung size.

Q33 preserves both:

- local ARA position for tracing the source crest and child pole;
- one common connected-relation energy unit for testing whether a recipient
  is actually half-sized in the source frame.

The source is the already-open Q27/Q28 public simulator cache. The later time
partition remains unchanged but is not fresh blind data.


## Methods

For connected relation matrix \(C_p(t)\):

\[
h_p=|\det C_p|^{1/3},\qquad
x_p=\frac{2h_p}{Q_{.95}^{dev}(h_p)},
\]

\[
E_p=\lVert C_p\rVert_F^2,\qquad
\rho_{c\mid p}=
\frac{Q_{.95}^{dev}(E_c)}{Q_{.95}^{dev}(E_p)},
\qquad
L=3+\rho.
\]

Sources begin at `x>=1.5`, release on the next slice, and lose energy from a
backward-traced crest. The two active endpoint relations are traced backward
eight slices to their latest local minima.

Frozen controls:

- two topology-matched non-endpoint relations;
- endpoint relations at seed `+37`;
- endpoint relations at time `+137` inside the same split.


In [1]:
from pathlib import Path
import json
import pandas as pd

ROOT = Path.cwd()
if not (ROOT / "Q33_TWO_AXIS_PARENT_CHILD_35_RESULTS.json").exists():
    ROOT = Path(r"F:\SystemFormulaFolder\GIT\ARA-GIT\analysis\quantum")

result = json.loads((ROOT / "Q33_TWO_AXIS_PARENT_CHILD_35_RESULTS.json").read_text())
validation = json.loads((ROOT / "Q33_TWO_AXIS_PARENT_CHILD_35_VALIDATION.json").read_text())
evaluation = result["splits"]["evaluation"]
print("Evaluation source events:", evaluation["source_events"])
print("Exact child routes:", evaluation["exact_child_routes"])
print("Verdict:", result["frozen_verdict"]["label"])
print("Independent validation:", validation["status"])


Evaluation source events: 11543
Exact child routes: 23086
Verdict: CROSS-RUNG 3.5 PROJECTION NOT SUPPORTED BY THIS IMPLEMENTATION
Independent validation: PASS


## Results

### Capacity, path and pole readings


In [2]:
exact = evaluation["routes"]["exact"]
headline = pd.DataFrame([
    {"reading": "energy capacity", "median": exact["event_mean_capacity_ratio"]["median"], "ARA target": 0.5},
    {"reading": "amplitude", "median": exact["event_mean_amplitude_ratio"]["median"], "ARA target": 0.5},
    {"reading": "determinant closure scale", "median": exact["event_mean_closure_scale_ratio"]["median"], "ARA target": 0.5},
    {"reading": "complete path", "median": exact["complete_path"]["median"], "ARA target": 3.5},
    {"reading": "child origin local x", "median": exact["child_origin_x"]["median"], "ARA target": 0.5},
    {"reading": "summed realised transfer", "median": exact["transfer_sum"]["median"], "ARA target": float("nan")},
])
headline


reading,median,ARA target
energy capacity,1.273490,0.5
amplitude,1.107934,0.5
determinant closure scale,1.145159,0.5
complete path,4.273490,3.5
child origin local x,0.041370,0.5
summed realised transfer,1.032651,NaN


The half-capacity reading fails under energy, amplitude and
determinant-closure scale. The pole-origin result succeeds strongly. The
transfer ratio is descriptive only because the simulator does not promise
local energy conservation.


### Branch stability


In [3]:
branch = pd.DataFrame([
    {
        "branch": label,
        "events": evaluation["branches"][label]["source_events"],
        "median capacity ratio": evaluation["branches"][label]["exact_event_mean_capacity_ratio"]["median"],
        "median child-origin x": evaluation["branches"][label]["exact_child_origin_x"]["median"],
    }
    for label in ("c2", "c4")
])
branch


branch,events,median capacity ratio,median child-origin x
c2,5772,1.245870,0.036746
c4,5771,1.306613,0.047042


### Relation-broken controls


In [4]:
controls = pd.DataFrame([
    {
        "control": control,
        "paired events": evaluation["routes"][control]["paired_events"],
        "median capacity ratio": evaluation["routes"][control]["event_mean_capacity_ratio"]["median"],
        "exact median-error advantage": result["evaluation_control_half_distance_advantage"][control],
        "bootstrap P(exact better)": result["evaluation_bootstrap"][control]["probability_exact_lower"],
    }
    for control in ("topology", "seed", "time")
])
controls


control,paired events,median capacity ratio,exact median-error advantage,bootstrap P(exact better)
topology,11543,1.362884,0.103850,1.0000
seed,10788,1.335173,0.071845,1.0000
time,10790,1.283664,0.010098,0.9395


Exact is more half-like than topology and seed controls, but
only 1.01% better than the time control with bootstrap probability 0.9395.
Both miss the frozen 5% and 0.95 gates.


### Geometry

![Q33 geometry](Q33_TWO_AXIS_PARENT_CHILD_35_GEOMETRY.png)

The upper panels show that the capacity distribution centers beyond a
same-sized ratio of `1`, not at half capacity. The lower-left panel shows that
backward child origins are nevertheless concentrated near the local `0` pole.


### Trial-level preview


In [5]:
trials = pd.read_csv(ROOT / "Q33_TWO_AXIS_PARENT_CHILD_35_TRIALS.csv")
trial_preview = trials[[
    "branch_label", "seed", "n_events", "exact_rho_mean",
    "topology_rho_mean", "seed_rho_mean", "time_rho_mean"
]].head(12)
trial_preview


branch_label,seed,n_events,exact_rho_mean,topology_rho_mean,seed_rho_mean,time_rho_mean
c2,0,46,1.208218,1.387779,1.411444,1.203752
c2,1,61,1.515168,1.837506,1.588677,1.512940
c2,2,60,1.817646,2.009751,1.343929,1.782934
c2,3,50,1.206777,1.270108,1.128762,1.168858
c2,4,51,1.489478,1.702070,1.211550,1.457314
c2,5,55,1.252689,1.198120,1.458286,1.236801
c2,6,45,1.247656,1.208109,1.302302,1.311590
c2,7,63,1.868876,1.946582,1.307944,1.877736
c2,8,58,1.462544,1.727857,1.493252,1.524639
c2,9,62,1.082516,1.159997,1.408583,1.050990


## Takeaways

1. The corrected quantum `3.5` construction has now been tested directly.
2. Q32's endpoint relations are valid ordered recipients but are not shown to
   be one rung below the source.
3. Temporal child order and octave child size are separate empirical claims.
4. The near-pole origin and near-unity median summed transfer survive and
   sharpen the handover account.
5. A future `3.5` test must choose the single boundary-nearest child, apply the
   fixed `1 -> 0.5` octave projection and test a consequence of that route.

## Reproduction and assumptions

Run `q33_two_axis_parent_child_35_test.py`, then
`q33_validate_two_axis_parent_child_35.py`. All development scales are frozen
from `t=0..249`. No future child value selects a route.

This simulator is exactly diagonal and is not hardware quantum data. The
result validates a computational crosswalk inside this source, not universal
quantum mechanics or the dark-sector ratio.
